<div style="background:#0d0d0d; padding:32px 40px; border-radius:12px; border-left:5px solid #7B61FF;">
<h1 style="color:#FFFFFF; font-family:monospace; margin:0 0 8px 0; font-size:26px;">
  🎲 Greedy vs. Sampling from Scratch
</h1>
<p style="color:#9E9E9E; font-size:14px; margin:0;">
  Decoding Complexities · companion notebook
</p>
</div>

---

### What this notebook covers

| Section | Concept |
|---|---|
| 1 | Load a small, free, instruction-tuned model — inference only, no training |
| 2 | Greedy decoding — same question, same answer, every time |
| 3 | Sampling — same question, different wording each time |
| 4 | Why this isn't a bug, and how to force reproducible output on demand |

We are not pre-deciding what the model will say. Whatever it actually outputs is what goes
into the final script, per this repo's narrative-evidence-alignment rule — we describe what
happened, not what we expected to happen.

### Recording note (confirmed 2026-10-07)

**This notebook is run once, in advance, and recorded as-is — not run live on screen during
filming.** Model load takes ~20s and each generation call ~1-3s; showing that dead time live
would break the "no dead canvas" rule this channel holds everywhere else. The video screen-shares
this notebook directly (its already-executed cells and outputs), not a separate slide, for the
demo section — so the captured outputs below are the actual source of truth for the script,
blog, and deck, not a re-typed summary of them.

**Final demo question, locked:** *"What's one benefit of drinking water first thing in the
morning?"* — do not paraphrase this later; every number/quote downstream traces back to this
exact wording.

**Pinned for reproducibility:** `Qwen/Qwen2.5-0.5B-Instruct`, `transformers==5.18.0`,
`torch==2.14.1`. Greedy decoding is only guaranteed deterministic for this exact model +
library combination — a different transformers/torch version could change wording.

In [4]:
# Section 1 -- load a small, free, instruction-tuned model. No training, inference only.
# Qwen2.5-0.5B-Instruct: ~1GB download, small enough to run on CPU in reasonable time,
# and actually trained to answer questions (unlike base GPT-2/DistilGPT-2).
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.eval()
print(f"Loaded {MODEL_NAME} -- {model.num_parameters():,} parameters.")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5467.64it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct -- 494,032,768 parameters.


In [8]:
def ask(question, do_sample, seed=None):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    if seed is not None:
        torch.manual_seed(seed)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=do_sample,
        temperature=1.0 if do_sample else None,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    reply_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply_ids, skip_special_tokens=True).strip()

QUESTION = "What's one benefit of drinking water first thing in the morning?"

# Section 2 -- greedy decoding (argmax): always takes the single highest-probability token
greedy_1 = ask(QUESTION, do_sample=False, seed=1)
greedy_2 = ask(QUESTION, do_sample=False, seed=2)

print(f"Greedy (run 1): \n \n {greedy_1} \n")
print(f"Greedy (run 2): \n \n {greedy_2} \n")
print(f"Identical: {greedy_1 == greedy_2}")

Greedy (run 1): 
 
 Drinking water first thing in the morning can offer several benefits for your health and overall well-being:

1. Hydration: Drinking water helps to keep you hydrated throughout the day, which is crucial for maintaining proper bodily functions. 

Greedy (run 2): 
 
 Drinking water first thing in the morning can offer several benefits for your health and overall well-being:

1. Hydration: Drinking water helps to keep you hydrated throughout the day, which is crucial for maintaining proper bodily functions. 

Identical: True


In [9]:
# Section 3 -- sampling: a weighted random draw over the same distribution
sample_1 = ask(QUESTION, do_sample=True)
sample_2 = ask(QUESTION, do_sample=True)
print(f"Sample 1: \n \n {sample_1} \n")
print(f"Sample 2: \n \n {sample_2} \n")
print(f"Identical: {sample_1 == sample_2}")

Sample 1: 
 
 One significant benefit of drinking water first thing in the morning is its restorative nature.

When we wake up earlier than usual or feel fatigue earlier in the day, it can be challenging to maintain our daily energy and performance levels during 

Sample 2: 
 
 Drinking water first thing in the morning can offer several benefits:

1. Upfront hydration: Drinking water first thing in the morning sets up your system with adequate magnesium levels, which helps regulate muscle cell function and enhances學習（ 

Identical: False


---
## Section 4 — Why This Isn't a Bug

A high-probability word winning most of the time, but not every time, isn't a flaw —
it's a deliberate design choice. Greedy decoding gives you the same output on every
call: predictable, but repetitive if you ask the same thing twice. Sampling gives you
natural variety: different wording, same meaning, every call — useful for conversation,
unhelpful if you need an exact repeat.

**If you need reproducible output** — automated tests, CI pipelines, demos that must
look identical every run — force greedy decoding (`do_sample=False`) or fix a random
seed before sampling. Both are one line in the same `generate()` call already shown
above.

**One caveat worth knowing, not building here:** push sampling further without any
filter on the probability tail, and small models in particular can drift into
genuinely low-quality, incoherent output — not just "different wording." Real systems
usually combine sampling with a filter on that tail before drawing. That's a separate
mechanism from decoding strategy itself, and a topic of its own.